# 03 — PINN in JAX

**Optional Track · Lê Nguyễn Ngọc Vũ**

This notebook reimplements the **same ODE PINN** from `module_3_pinn/notebooks/01_pinn_ode.ipynb` using JAX + Flax.

**Target ODE:** Exponential decay
```
du/dt = -k * u,   u(0) = 1,   k = 1.0
Analytic: u(t) = exp(-t)
```

**Goal:** See how JAX's `vmap(grad(...))` makes the PINN residual cleaner and faster than the PyTorch equivalent.

---

In [ ]:
import jax
import jax.numpy as jnp
import flax.linen as nn
import optax
import numpy as np
import matplotlib.pyplot as plt

key = jax.random.PRNGKey(42)
k = 1.0
T_END = 5.0
print(f'JAX devices: {jax.devices()}')

## 1. PINN Model (Flax)

In [ ]:
class PINN(nn.Module):
    hidden: int = 64
    n_layers: int = 3

    @nn.compact
    def __call__(self, t):
        """t: scalar (single time point)."""
        x = t.reshape(1)  # (1,)
        for _ in range(self.n_layers):
            x = nn.Dense(self.hidden)(x)
            x = nn.tanh(x)
        return nn.Dense(1)(x)[0]  # scalar output

model = PINN()
key, init_key = jax.random.split(key)
params = model.init(init_key, jnp.array(0.0))['params']
print('PINN initialized.')

## 2. Physics Residual — The JAX Way

In PyTorch we needed `autograd.grad(..., create_graph=True)` with careful shape management.

In JAX:
- `u(t)` is a plain Python function (single scalar input)
- `du/dt` = `jax.grad(u)(t)` — one line
- Vectorize over collocation points with `vmap`

In [ ]:
def u(params, t_scalar):
    """Network output for a single scalar time point."""
    return model.apply({'params': params}, t_scalar)

def residual_single(params, t_scalar, k):
    """ODE residual du/dt + k*u at a single point."""
    du_dt = jax.grad(u, argnums=1)(params, t_scalar)  # scalar
    return du_dt + k * u(params, t_scalar)

# Vectorize over collocation points
residual_batch = jax.vmap(residual_single, in_axes=(None, 0, None))

# Test with random collocation points
key, ck = jax.random.split(key)
t_test = jax.random.uniform(ck, (5,)) * T_END
res = residual_batch(params, t_test, k)
print(f'Residuals at init (should be ~random): {res}')

## 3. Loss Function

In [ ]:
def loss_fn(params, t_colloc, k, w_physics=1.0, w_ic=10.0):
    # Physics loss
    res = residual_batch(params, t_colloc, k)
    loss_physics = jnp.mean(res ** 2)

    # Initial condition loss: u(0) = 1
    u_at_0 = u(params, jnp.array(0.0))
    loss_ic = (u_at_0 - 1.0) ** 2

    return w_physics * loss_physics + w_ic * loss_ic, (loss_physics, loss_ic)

# Wrap so jax.grad only sees the loss scalar
def loss_scalar(params, t_colloc):
    total, _ = loss_fn(params, t_colloc, k)
    return total

# Test
total, (lp, lic) = loss_fn(params, t_test, k)
print(f'Initial total loss: {total:.4f}  physics: {lp:.4f}  ic: {lic:.4f}')

## 4. Training Loop

In [ ]:
N_COLLOC = 1000
N_STEPS = 10000

optimizer = optax.adam(learning_rate=1e-3)
opt_state = optimizer.init(params)

# Sample collocation points once
key, ck = jax.random.split(key)
t_colloc = jax.random.uniform(ck, (N_COLLOC,)) * T_END

@jax.jit
def train_step(params, opt_state, t_colloc):
    (total, (lp, lic)), grads = jax.value_and_grad(
        loss_fn, has_aux=True
    )(params, t_colloc, k)
    updates, new_opt_state = optimizer.update(grads, opt_state, params)
    new_params = optax.apply_updates(params, updates)
    return new_params, new_opt_state, total, lp, lic

history = {'total': [], 'physics': [], 'ic': []}

for step in range(N_STEPS):
    params, opt_state, total, lp, lic = train_step(params, opt_state, t_colloc)
    history['total'].append(float(total))
    history['physics'].append(float(lp))
    history['ic'].append(float(lic))

    if (step + 1) % 1000 == 0:
        print(f'Step {step+1:5d}/{N_STEPS}  '
              f'Total: {total:.6f}  Physics: {lp:.6f}  IC: {lic:.6f}')

## 5. Compare with Analytic Solution

In [ ]:
t_eval = jnp.linspace(0, T_END, 500)

# vmap over evaluation points
u_batch = jax.vmap(u, in_axes=(None, 0))
u_pred = np.array(u_batch(params, t_eval))
u_analytic = np.exp(-k * np.array(t_eval))

plt.figure(figsize=(8, 4))
plt.plot(t_eval, u_analytic, 'b-', linewidth=2, label='Analytic: exp(-t)')
plt.plot(t_eval, u_pred, 'r--', linewidth=2, label='JAX PINN')
plt.xlabel('t')
plt.ylabel('u(t)')
plt.title('JAX PINN vs Analytic (du/dt = -ku)')
plt.legend()
plt.grid(True)
plt.show()

mse = np.mean((u_pred - u_analytic) ** 2)
print(f'MSE vs analytic: {mse:.2e}')

## 6. Plot Loss History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogy(history['physics'], label='Physics loss')
axes[0].semilogy(history['ic'], label='IC loss')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss (log scale)')
axes[0].set_title('Loss components')
axes[0].legend()
axes[0].grid(True)

axes[1].semilogy(history['total'])
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Total loss (log scale)')
axes[1].set_title('Total loss')
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 7. JAX vs PyTorch PINN Code Comparison

| Step | PyTorch (`01_pinn_ode.ipynb`) | JAX (this notebook) |
|------|-------------------------------|---------------------|
| `du/dt` at one point | `autograd.grad(u, t, grad_outputs=ones, create_graph=True)` | `jax.grad(u, argnums=1)(params, t)` |
| `du/dt` over batch | Inside loop OR manual `vmap` workaround | `jax.vmap(jax.grad(u, argnums=1), in_axes=(None, 0))(params, t_batch)` |
| JIT compile | `torch.compile(model)` | `@jax.jit` on entire train step |
| Loss + gradient | `loss.backward()` | `jax.value_and_grad(loss_fn, has_aux=True)` |
| Second derivative | Nested `autograd.grad` with `create_graph=True` | `jax.grad(jax.grad(u))` — same syntax |

**Key advantage of JAX for PINNs:** `jax.vmap(jax.grad(...))` is efficient, composable, and doesn't require `create_graph=True` or graph retention tricks.


## 8. Extension: PDE in JAX

*(Optional — extend to heat equation if time permits)*

In [ ]:
# --- TODO: Extend to the 1D heat equation ---
# Model: u(params, x_scalar, t_scalar) -> scalar
# Residual: u_t - alpha * u_xx
#
# Compute:
#   u_t = jax.grad(u, argnums=2)(params, x, t)
#   u_x = jax.grad(u, argnums=1)(params, x, t)
#   u_xx = jax.grad(lambda x: jax.grad(u, argnums=1)(params, x, t))(x)
#
# Then vmap over (x, t) collocation pairs
